# Grain-boundary size field — 2D conformal mesh

Gmsh uses a distance Threshold field: element size is `mesh_size_gb` on GB
curves and grows to `mesh_size_bulk` in the grain interior
(`DistMin = mesh_size_gb`, `DistMax = 2 * mesh_size_bulk`).

This notebook meshes the same polygons twice: almost uniform (`gb ≈ bulk`)
versus GB-refined (`gb << bulk`).

Requires `gmsh` (`pip install upxo[mesh]`). `confMesh2d` (pygmsh) is deprecated.
Canonical mesh-only demo: `confMesh2d_gmsh.ipynb`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import box

from upxo.meshing.gsmesh2d import mesh_gs
from upxo.meshing.writer_ABQ import summarize_inp


In [ ]:
cells = {
    1: box(0, 0, 2, 2),
    2: box(2, 0, 4, 2),
    3: box(0, 2, 2, 4),
    4: box(2, 2, 4, 4),
}

In [ ]:
uniform = mesh_gs(cells, mesh_size_gb=0.6, mesh_size_bulk=0.6,
                  mesh_algo=6, recombine_to_quads=False)
refined = mesh_gs(cells, mesh_size_gb=0.2, mesh_size_bulk=0.9,
                  mesh_algo=6, recombine_to_quads=False)
for name, r in (('uniform', uniform), ('refined', refined)):
    r['mesher'].form_elsets_gmsh()
    r['mesher'].build_boundary_nsets(); r['mesher'].build_gb_nset()
    print(name, 'n_tri', r['n_tri'], 'n_nodes', r['n_nodes'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), dpi=120)
uniform['mesher'].plot_by_grain(ax=axes[0], show_gb=True, show_nsets=False,
                                title='gb = bulk = 0.6')
refined['mesher'].plot_by_grain(ax=axes[1], show_gb=True, show_nsets=False,
                               title='gb = 0.2, bulk = 0.9')
fig.tight_layout()
fig

In [ ]:
out = Path.cwd() / 'confMesh2d_size_field_out'
p_u = uniform['mesher'].export_abaqus_inp(out / 'uniform_cps3.inp', plane='stress')
p_r = refined['mesher'].export_abaqus_inp(out / 'refined_cps3.inp', plane='stress')
p_u, summarize_inp(p_u), p_r, summarize_inp(p_r)